# Observing Multi-Step Tool Tracing

A single LLM call is easy to debug. But what happens when your Agent has tools? 

The process looks like this:
1. You send a prompt.
2. LLM thinks "I need to use a tool."
3. Pydantic runs your Python function.
4. The result goes back to the LLM.
5. LLM generates the final answer.

This is heavily prone to failure. Logfire traces this entire "waterfall" beautifully.

In [5]:
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv

# Load .env first so logfire finds the LOGFIRE_API_KEY!
load_dotenv()

import logfire
from pydantic_ai import Agent

logfire.configure()
logfire.instrument_pydantic_ai()
logfire.info('Logfire tool tracing connected! Hello {place}!', place='World')

# 1. Setup our Agent
weather_agent = Agent(
    'groq:llama-3.3-70b-versatile',
    system_prompt="You are a helpful travel assistant."
)

# 2. Setup our Tool
@weather_agent.tool_plain
def check_flight_status(flight_number: str) -> str:
    """
    Use this to check if a flight is delayed or on-time.
    """
    print(f"\n[System Alert] Checking database for flight {flight_number}...")
    if "777" in flight_number:
        return "Delayed by 2 hours due to thunderstorms."
    return "On-time."

Logfire project URL: ]8;id=257992;https://logfire-us.pydantic.dev/d-hackmt/alia\https://logfire-us.pydantic.dev/d-hackmt/alia]8;;\

13:47:11.855 Logfire tool tracing connected! Hello World!


### Running the trace

In [6]:
print("Asking the Agent...")
response = weather_agent.run_sync("Is my flight BA777 delayed?")

print("\n--- Agent Response ---")
print(response.output)

Asking the Agent...
13:47:13.627 weather_agent run
13:47:13.630   chat llama-3.3-70b-versatile
13:47:13.984   running 1 tool
13:47:13.985     running tool: check_flight_status

[System Alert] Checking database for flight BA777...
13:47:13.987   chat llama-3.3-70b-versatile
13:47:14.370   running 1 tool
13:47:14.371     running tool: check_flight_status

[System Alert] Checking database for flight BA777...
13:47:14.373   chat llama-3.3-70b-versatile

--- Agent Response ---
I've checked the status of your flight BA777, and it's currently delayed by 2 hours due to thunderstorms. I recommend checking with the airport or the airline for the most up-to-date information and to see if there are any other travel options available.


### Viewing the Result

Open your Logfire dashboard. You will see a deeply nested, color-coded trace:

- 🟢 **Complete Run**
  - 🔵 **Model Completion 1**: The LLM replying with a "Call check_flight_status('BA777')" command instead of a normal string.
  - 🟡 **Tool Execution**: Tracing exactly how many milliseconds the python function took to run.
  - 🔵 **Model Completion 2**: The LLM receiving the output string and converting it into human-readable text.

This allows you to find bottlenecks (is the API slow, or is the LLM slow?) instantly!